# Native HHL Workshop

Solve the original four-dimensional system with a four-bit fractional phase counter. Every source code stage is retained, including preparation, finite-resolution inverse, full complex state, postselection, rounding, fidelity and the final plot. The original source uses NumPy/SciPy; this version computes the same matrix exponential, solve and norms with the existing PyTorch dependency. No provider account is required.


In [ ]:
# The installed FlagQuantum environment supplies the public synthesis APIs.
# No provider authentication or notebook-time package installation is needed.


## Original matrix and normalized vector

Keep all matrix entries and b=[1,2,4,3]. The matrix must be symmetric with eigenvalues strictly in (0,1), and b is normalized before preparation.


In [ ]:
import math
from pathlib import Path
from typing import Annotated

import torch
from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.tutorials.workshops.hhl_workshop.overview.workflow import (
    StateCheck,
    extract_solution,
    plot_solution,
    validate_problem,
    verify_program,
)
from flagquantum import synth
from flagquantum.compiler import ResourceEstimate
from flagquantum.synth import (
    Output,
    QBit,
    QNum,
    allocate,
    apply,
    prepare_amplitudes,
    qfunc,
    qpe,
    select_rotation,
    unitary,
    within,
)

A = torch.tensor(
    [
        [0.28, -0.01, 0.02, -0.1],
        [-0.01, 0.5, -0.22, -0.07],
        [0.02, -0.22, 0.43, -0.05],
        [-0.1, -0.07, -0.05, 0.42],
    ],
    dtype=torch.float64,
)
b = torch.tensor([1.0, 2.0, 4.0, 3.0], dtype=torch.float64)
b = b / torch.linalg.vector_norm(b)
validate_problem(A, b)
print("A =", A, "\nb =", b)
w = torch.linalg.eigvalsh(A)
print("Eigenvalues:", w)
sol_classical = torch.linalg.solve(A, b)
print("Classical solution:", sol_classical)
num_qubits = 2
checks: list[StateCheck] = []


## Prepare the right-hand side

The preparation helper is itself a native root qfunc. Its bound real array is normalized above, rather than replacing the original vector with rounded constants.


In [ ]:
@qfunc
def load_b(amplitudes: list[float], res: Output[Annotated[QNum, 2]]) -> None:
    prepare_amplitudes(amplitudes, 0.0, res)


In [ ]:
qprog_b_load = synth.synthesize(
    load_b, classical_inputs={"amplitudes": b.tolist()}, max_workspace=0
)
checks.append(verify_program(qprog_b_load, preparation_only=True))
print(qprog_b_load.report["resources"])


In [ ]:
prepared = fq.run(qprog_b_load).statevector()
assert isinstance(prepared, torch.Tensor)


In [ ]:
print("Original b:", b)
print("Prepared amplitudes:", prepared.flatten())
torch.testing.assert_close(
    prepared.flatten(), b.to(torch.complex128), atol=1e-12, rtol=0
)


## Reciprocal-amplitude table

For each four-bit raw phase word k, the fractional value is k/16. The source amplitude is zero for k=0 and (1/16)/(k/16)=1/k otherwise. A selected Y rotation with angle `2 * asin(1/k)` implements that amplitude. The root allocates the indicator before entering the reversible region; this helper borrows it without allocating or resetting.


In [ ]:
QPE_SIZE = 4
ANGLES = tuple(
    0.0 if word == 0 else 2 * math.asin(1 / word) for word in range(2**QPE_SIZE)
)


@qfunc
def simple_eig_inv(phase_: QNum, indicator: QBit) -> None:
    select_rotation("Y", ANGLES, phase_, indicator)


## Native HHL composition

The source generic matrix callback is fixed here to its actual matrix exp(2*pi*i*A). Native within/apply generates inverse QPE around the reciprocal rotation. Preparation is expressed directly in the root because the current parameterized helper contract admits borrowed writable ports, not a fresh Output allocation. This is an explicit fixed-program translation; it does not claim generic QCallable support.


In [ ]:
@qfunc
def main(
    amplitudes: list[float],
    res: Output[Annotated[QNum, 2]],
    phase_: Output[Annotated[QNum, (4, False, 4)]],
    indicator: Output[QBit],
) -> None:
    prepare_amplitudes(amplitudes, 0.0, res)
    allocate(phase_)
    allocate(indicator)
    with within():
        qpe(lambda: unitary(UNITARY, res), phase_)
    with apply():
        simple_eig_inv(phase_, indicator)


In [ ]:
unitary_tensor = torch.matrix_exp(2j * math.pi * A)
UNITARY = tuple(
    tuple(complex(value) for value in row) for row in unitary_tensor.tolist()
)
print("Source phase precision:", QPE_SIZE)


In [ ]:
qprog_hhl = synth.synthesize(
    main,
    classical_inputs={"amplitudes": b.tolist()},
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
checks.append(verify_program(qprog_hhl))


## Inspect actual resources

The public resource report replaces the hosted viewer. Depth and operation counts describe synthesized instructions before Runtime basis conversion; they are not CX/U hardware-basis claims.


In [ ]:
print(qprog_hhl.report["resources"])


In [ ]:
resource = qprog_hhl.report["resources"]
assert isinstance(resource, ResourceEstimate)
print("Synthesized instruction depth:", resource.depth)


In [ ]:
print("Complete resource estimate:", qprog_hhl.report["resources"])


In [ ]:
result = fq.run(qprog_hhl)
state = result.statevector()
assert isinstance(state, torch.Tensor)
print("Full complex state:", state.flatten())


In [ ]:
print("Named registers:", dict(qprog_hhl.registers))


In [ ]:
print(
    "Decoded joint probabilities:",
    dict(qprog_hhl.probabilities(result, registers=("res", "phase_", "indicator"))),
)


In [ ]:
print(
    "The readout uses named res,phase_,indicator ports; no physical-wire decoding is needed."
)


## Postselect and round exactly as the source

Select indicator=1 and phase_=0,keep amplitudes above 1e-10,divide by C=1/16,and round real and imaginary parts to five decimals. The retained phase register need not be zero on every branch: phase leakage is physical finite-precision HHL residue,not dirty compiler work. Report the actual probability of the selected branch.


In [ ]:
solution = extract_solution(qprog_hhl)
qsol = solution.rounded_quantum
print("Quantum solution:", qsol.abs() / torch.linalg.vector_norm(qsol))
print("Classical solution:", sol_classical / torch.linalg.vector_norm(sol_classical))
print("Postselection probability:", solution.postselection_probability)
print("Indicator probability:", solution.indicator_probability)
print("Retained phase leakage:", solution.retained_phase_leakage_probability)


In [ ]:
print("Solution fidelity:", solution.fidelity)


## Preserve the source magnitude diagnostic honestly

The source calls angle(qsol) a global phase but removes a different phase from each entry. We retain its numerical diagnostic and plot as a magnitude comparison; this is not a physical global-phase correction. Full complex state comparison and complex relative distance are reported separately.


In [ ]:
qsol_corrected = solution.magnitude_diagnostic
print("Classical:", sol_classical)
print("HHL source magnitude diagnostic:", qsol_corrected)
print(
    "Source magnitude relative distance (%):",
    solution.magnitude_relative_distance_percent,
)
print("Complex relative distance:", solution.complex_relative_distance)


In [ ]:
plot_path = plot_solution(solution, Path("artifacts/development/hhl_workshop"))
print("Saved comparison:", plot_path)

display(Image(filename=str(plot_path)))


No matrix size,phase precision,source vector or reciprocal-table entry was reduced. Full-state validation includes absolute phase and all compiler workspace. Finite-resolution postselection success and residual phase are reported; no exact linear solve,hardware execution,resource optimality or quantum speedup is claimed.
